# Pattern Portal Real-Data Lab

A compact notebook companion for the Real-Data Cases page. Use it as a starting point for classification, regression, time-series forecasting, and market backtesting workflows.

## Install Dependencies
Run the install cell if your environment does not already include the listed packages.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib yfinance -q

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## Case 1: Housing Regression
Pipeline: load data, split, train a baseline, report MAE/RMSE/R2, and inspect the largest errors.

In [ ]:
data = fetch_california_housing(as_frame=True)
X = data.frame.drop(columns=['MedHouseVal'])
y = data.frame['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

model = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE:', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))
print('R2:', r2_score(y_test, pred))
pd.Series(np.abs(y_test - pred), index=y_test.index).sort_values(ascending=False).head()

## Case 2: Fraud Classification Template
Download the Kaggle credit-card fraud CSV, place it next to this notebook as `creditcard.csv`, then run the cell. The important part is the evaluation: precision, recall, PR-AUC, and the confusion matrix at a chosen threshold.

In [ ]:
# df = pd.read_csv('creditcard.csv')
# X = df.drop(columns=['Class'])
# y = df['Class']
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
# )
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
# model = RandomForestClassifier(
#     n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
# )
# model.fit(X_train, y_train)
# proba = model.predict_proba(X_test)[:, 1]
# pred = (proba >= 0.35).astype(int)
# print(confusion_matrix(y_test, pred))
# print(classification_report(y_test, pred, digits=3))
# print('PR-AUC:', average_precision_score(y_test, proba))

## Case 3: Time-Series Forecast Template
Use this structure for energy-demand data: sort by time, create lag features from the past only, split by time, and compare against a naive baseline.

In [ ]:
# Replace this synthetic series with your real timestamped demand series.
rng = np.random.default_rng(RANDOM_STATE)
idx = pd.date_range('2022-01-01', periods=500, freq='D')
daily = pd.DataFrame({
    'y': 50 + np.linspace(0, 8, len(idx)) + 8 * np.sin(np.arange(len(idx)) / 7) + rng.normal(0, 2, len(idx))
}, index=idx)

for lag in [1, 2, 7, 14]:
    daily[f'lag_{lag}'] = daily['y'].shift(lag)
daily['rolling_7'] = daily['y'].shift(1).rolling(7).mean()
daily = daily.dropna()

split = int(len(daily) * 0.8)
train, test = daily.iloc[:split], daily.iloc[split:]
features = [c for c in daily.columns if c != 'y']

model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
model.fit(train[features], train['y'])
pred = model.predict(test[features])
naive = test['lag_1']

print('Model MAE:', mean_absolute_error(test['y'], pred))
print('Naive MAE:', mean_absolute_error(test['y'], naive))

## Case 4: Market Backtest Template
Market indicators are heuristic until validated. Include costs, compare to buy-and-hold, and inspect drawdown.

In [ ]:
yf = __import__('yfinance')

px = yf.download('SPY', start='2015-01-01', auto_adjust=True, progress=False)
close = px['Close']
returns = close.pct_change()

fast = close.rolling(20).mean()
slow = close.rolling(100).mean()
signal = (fast > slow).astype(int).shift(1).fillna(0)

turnover = signal.diff().abs().fillna(0)
cost = turnover * 0.0005
strategy = signal * returns - cost
equity = (1 + strategy.fillna(0)).cumprod()
drawdown = equity / equity.cummax() - 1
sharpe = strategy.mean() / strategy.std() * np.sqrt(252)

print('Sharpe:', sharpe)
print('Max drawdown:', drawdown.min())
print('Annual turnover:', turnover.mean() * 252)